# Meme Decoder Colab Pipeline

This notebook runs the MemeCap ablation pipeline on Google Colab:

- Input settings `1-5`
- Training strategies: `zero-shot`, `projector-only`, `projector-lora`
- Google Drive data/cache/output paths

Recommended first run: use the smoke-test cells with small sample counts before launching full training.

## 1. Runtime

In Colab, choose **Runtime -> Change runtime type -> GPU**. L4/A100 is best; T4 can be used with batch size 1 and gradient accumulation.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Basic GPU check
!nvidia-smi

Thu Apr 30 06:13:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Or Update The Repository

In [5]:
import os
from pathlib import Path

REPO_URL = "https://github.com/baoyunfan0101/meme-decoder.git"
REPO_DIR = Path("/content/meme-decoder")

if REPO_DIR.exists():
    %cd /content/meme-decoder
    !git pull
else:
    %cd /content
    !git clone $REPO_URL
    %cd /content/meme-decoder

/content
Cloning into 'meme-decoder'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 48 (delta 16), reused 47 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 32.75 KiB | 16.37 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/meme-decoder


## 3. Install Dependencies

In [ ]:
%cd /content/meme-decoder
!pip install -q -U "numpy>=2.0,<2.6"
!pip install -q -r requirements.txt

# Qwen2.5-VL support is actively updated in transformers.
!pip install -q -U transformers accelerate peft

/content/meme-decoder
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 125.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 150.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 63.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 46.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behav

## 4. Configure Data Paths

Expected data layout in Drive:

```text
/content/drive/MyDrive/meme-decoder-data/data/
  raw/
    memes-trainval.json
    memes-test.json
    memes/
      memes/
        memes_bpet7l.png
        ...
  processed/
    memes-trainval.ocr.json
    memes-test.ocr.json
```

Set `RAW_DIR` to `data/raw`, not to `data/raw/memes/memes`. `src/path_utils.py` automatically descends into nested `memes/` folders when locating images.

Because images are already uploaded to Drive, the main training/evaluation commands below do not use `--allow-download`.


In [ ]:
from pathlib import Path
import os

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/meme-decoder-data")
DATA_DIR = DRIVE_PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
IMAGE_CACHE_DIR = RAW_DIR / "memes" / "memes"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["RAW_DIR"] = str(RAW_DIR)
os.environ["PROCESSED_DIR"] = str(PROCESSED_DIR)
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)

print("RAW_DIR        =", RAW_DIR)
print("IMAGE_CACHE_DIR=", IMAGE_CACHE_DIR)
print("PROCESSED_DIR  =", PROCESSED_DIR)
print("OUTPUT_DIR     =", OUTPUT_DIR)
print("image cache exists:", IMAGE_CACHE_DIR.exists())


## 5. Optional: Download The Shared Google Drive Data

Run this only if you have not already placed the data under `DRIVE_PROJECT_DIR/data`.

In [ ]:
%cd /content/meme-decoder

trainval_path = PROCESSED_DIR / "memes-trainval.ocr.json"
test_path = PROCESSED_DIR / "memes-test.ocr.json"

if trainval_path.exists() and test_path.exists():
    print("Processed data already exists; skipping folder download.")
else:
    !python -m scripts.download_data --output "$DRIVE_PROJECT_DIR"

## 6. Verify Dataset Counts

In [ ]:
import json

for path in [PROCESSED_DIR / "memes-trainval.ocr.json", PROCESSED_DIR / "memes-test.ocr.json"]:
    print(path)
    if not path.exists():
        print("  MISSING")
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    print("  count:", len(data))
    print("  keys :", sorted(data[0].keys()) if data else [])

## 7. Verify Local Meme Images

Your images are already in Google Drive under `data/raw/memes/memes/`, so normally you should only verify that the files referenced by `img_fname` exist.

Run the check below before training. If it reports many missing files, fix the Drive upload path before using URL download.


In [ ]:
import json
from pathlib import Path

def check_images(json_path, limit=20):
    records = json.loads(Path(json_path).read_text(encoding="utf-8"))
    missing = []
    checked = 0
    for record in records[:limit]:
        fname = record.get("img_fname", "")
        path = IMAGE_CACHE_DIR / fname
        checked += 1
        if not path.exists():
            missing.append(str(path))
    print(f"Checked {checked} examples from {Path(json_path).name}")
    print(f"Missing: {len(missing)}")
    for path in missing[:10]:
        print("  missing:", path)

check_images(PROCESSED_DIR / "memes-trainval.ocr.json", limit=20)
check_images(PROCESSED_DIR / "memes-test.ocr.json", limit=20)


In [ ]:
# Optional fallback only: run this if the local image check above reports missing files.
# This downloads images from each record's url and caches them under RAW_DIR.
# %cd /content/meme-decoder
# !python -m scripts.download_images --processed-dir "$PROCESSED_DIR" --raw-dir "$RAW_DIR"


## 8. Zero-Shot Smoke Test

Input setting `4` means: image + title + image captions + OCR.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode eval \
  --strategy zero-shot \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --eval-json memes-test.ocr.json \
  --eval-max-samples 20


## 9. Projector-Only Smoke Training

This automatically creates 5 fold files from `memes-trainval.ocr.json` if they do not exist.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-only \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum-steps 8 \
  --eval-max-samples 20


## 10. Projector + LoRA Smoke Training

For T4 GPUs, keep `batch-size=1`, use gradient accumulation, and start with `lora-r=8`.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-lora \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum-steps 8 \
  --lora-r 8 \
  --eval-max-samples 20


## 11. Full Training Examples

Run these after the smoke tests pass.

In [ ]:
# Full projector-only training, setting 4
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-only \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --epochs 2 \
  --batch-size 1 \
  --grad-accum-steps 8


In [ ]:
# Full projector + LoRA training, setting 4
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-lora \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --epochs 2 \
  --batch-size 1 \
  --grad-accum-steps 8 \
  --lora-r 8


## 12. Evaluate A Trained Checkpoint

Use the checkpoint list cell to find the run name, then set `MODEL_PATH`.

In [ ]:
from pathlib import Path

ckpt_root = OUTPUT_DIR / "checkpoints"
if ckpt_root.exists():
    for path in sorted(ckpt_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)[:10]:
        print(path)
else:
    print("No checkpoints yet:", ckpt_root)

In [ ]:
# Change this to one of the paths printed above.
MODEL_PATH = str(OUTPUT_DIR / "checkpoints" / "<run_name>")
STRATEGY = "projector-lora"  # or "projector-only"
INPUT_SETTING = 4

%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode eval \
  --strategy "$STRATEGY" \
  --input-setting "$INPUT_SETTING" \
  --model-path "$MODEL_PATH" \
  --checkpoint-name best \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --eval-json memes-test.ocr.json


## 13. Optional: Zero-Shot Input Ablation 1-5

In [ ]:
%cd /content/meme-decoder
for setting in [1, 2, 3, 4, 5]:
    print("\n=== zero-shot setting", setting, "===")
    !python -m scripts.run_pipeline \
      --mode eval \
      --strategy zero-shot \
      --input-setting "$setting" \
      --processed-dir "$PROCESSED_DIR" \
      --raw-dir "$RAW_DIR" \
      --output-dir "$OUTPUT_DIR" \
      --eval-json memes-test.ocr.json \
      --eval-max-samples 50
